In [12]:
import pandas as pd
pd.set_option('display.max_columns', None)

dir_path = 'data/processed/fiis_consolidados/'
fii = 'HABITAT II - FUNDO DE INVESTIMENTO IMOBILIÁRIO/'
file = 'fii_mensal_consolidado_.parquet'

df = pd.read_parquet(dir_path + fii + file)

In [18]:
df['Percentual_Dividend_Yield_Mes'] = pd.to_numeric(df['Percentual_Dividend_Yield_Mes'], errors='coerce')
df[ (df['Percentual_Dividend_Yield_Mes'] > 0) ][['Data_Referencia','Percentual_Dividend_Yield_Mes']]

,Data_Referencia,Percentual_Dividend_Yield_Mes
4530,2019-10-01,0.010940
4531,2019-11-01,0.007874
4532,2019-12-01,0.007874
4675,2020-01-01,0.009360
4676,2020-02-01,0.008565
...,...,...
3954,2025-07-01,0.011147
3956,2025-09-01,0.010046
3957,2025-10-01,0.010054
3958,2025-11-01,0.010026


In [4]:
df.columns

Index(['CNPJ_Fundo_Classe', 'Data_Referencia', 'Versao',
       'Total_Necessidades_Liquidez', 'Disponibilidades', 'Titulos_Publicos',
       'Titulos_Privados', 'Fundos_Renda_Fixa', 'Total_Investido',
       'Direitos_Bens_Imoveis', 'Terrenos', 'Imoveis_Renda_Acabados',
       'Imoveis_Renda_Construcao', 'Imoveis_Venda_Acabados',
       'Imoveis_Venda_Construcao', 'Outros_Direitos_Reais', 'Acoes',
       'Debentures', 'Bonus_Subscricao',
       'Certificados_Deposito_Valores_Mobiliarios', 'Cedulas_Debentures',
       'Fundo_Acoes', 'FIP', 'FII', 'FDIC', 'Outras_Cotas_FI',
       'Notas_Promissorias', 'Acoes_Sociedades_Atividades_FII',
       'Cotas_Sociedades_Atividades_FII', 'CEPAC', 'CRI', 'CRI_CRA',
       'Letras_Hipotecarias', 'LCI', 'LCI_LCA', 'LIG',
       'Outros_Valores_Mobliarios', 'Valores_Receber',
       'Contas_Receber_Aluguel', 'Contas_Receber_Venda_Imoveis',
       'Outros_Valores_Receber', 'Rendimentos_Distribuir',
       'Taxa_Administracao_Pagar', 'Taxa_Performance_

In [6]:
import os 

path = 'data/processed/fiis_consolidados/'
pastas_fii = os.listdir(path)

dict_cnpj = {}
for pasta_fii in pastas_fii:
    try:
        file = '/fii_mensal_consolidado_.parquet'
        cnpj = pd.read_parquet(path + pasta_fii + file, columns=['CNPJ_Fundo_Classe'])
        dict_cnpj[pasta_fii] = cnpj['CNPJ_Fundo_Classe'].values[0]
    except:
        sub_pasta = os.listdir(path + pasta_fii)
        if sub_pasta:
            try:
                file = '/fii_mensal_consolidado_.parquet'
                cnpj = pd.read_parquet(path + pasta_fii + sub_pasta[0] + file, columns=['CNPJ_Fundo_Classe'])
                dict_cnpj[pasta_fii] = cnpj['CNPJ_Fundo_Classe'].values[0]
            except:
                print(pasta_fii)


In [9]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://fnet.bmfbovespa.com.br/fnet/publico"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": f"{BASE_URL}/abrirGerenciadorDocumentosCVM",
}

def get_status_fundo(cnpj: str) -> dict:
    cnpj_limpo = "".join(filter(str.isdigit, cnpj))
    
    # Sessão isolada por fundo (importante para o isSession funcionar)
    session = requests.Session()
    session.headers.update(HEADERS)
    
    try:
        # ── Passo 1: seta o CNPJ na sessão do servidor ────────────────────────
        session.get(
            f"{BASE_URL}/abrirGerenciadorDocumentosCVM",
            params={"cnpjFundo": cnpj_limpo},
            timeout=10,
        )

        # ── Passo 2: busca os dados via AJAX ──────────────────────────────────
        resp = session.get(
            f"{BASE_URL}/pesquisarGerenciadorDocumentosDados",
            params={
                "d": 1,
                "s": 0,
                "l": 1,                        # só 1 registro é suficiente
                "o[0][dataReferencia]": "desc",
                "idCategoriaDocumento": 0,
                "idTipoDocumento": 0,
                "idEspecieDocumento": 0,
                "isSession": "true",
                "_": int(time.time() * 1000),
            },
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()

        registros = data.get("data", [])
        total = data.get("recordsTotal", 0)

        if not registros or total == 0:
            return {"cnpj": cnpj, "nome": None, "situacao": "SEM_DOCUMENTOS"}

        primeiro = registros[0]
        return {
            "cnpj": cnpj,
            "nome": primeiro.get("descricaoFundo"),
            "situacao": primeiro.get("situacaoDocumento"),   # "A" ou "C"
            "descricao_status": primeiro.get("descricaoStatus"),
            "ticker": primeiro.get("nomePregao"),
            "total_docs": total,
        }

    except Exception as e:
        return {"cnpj": cnpj, "nome": None, "situacao": "ERRO", "descricao_status": str(e)}


# ── Lote com ThreadPoolExecutor ───────────────────────────────────────────────
def processar_lote(lista_cnpjs: list, max_workers: int = 5) -> pd.DataFrame:
    resultados = []
    total = len(lista_cnpjs)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_status_fundo, cnpj): cnpj for cnpj in lista_cnpjs}
        for i, future in enumerate(as_completed(futures), 1):
            r = future.result()
            resultados.append(r)
            print(f"[{i}/{total}] {r['cnpj']} → {r['situacao']} | {r.get('nome', '')}")

    df = pd.DataFrame(resultados)
    print("\n── Resumo ──────────────────────")
    print(df["situacao"].value_counts())
    return df


k = pd.DataFrame(list(dict_cnpj.items()), columns=["FII", "CNPJ"])
lista_cnpjs = k['CNPJ'].tolist() 
kk = processar_lote(lista_cnpjs)

[1/1340] 59.680.687/0001-04 → A | TANE FZMG FUNDO DE INVESTIMENTO EM CADEIAS PRODUTIVAS AGROINDUSTRIAIS IMOBILIÁRIO
[2/1340] 43.101.312/0001-02 → A | MEIRINGEN FUNDO DE INVESTIMENTO IMOBILIÁRIO
[3/1340] 48.565.175/0001-72 → A | TANE FLEM FUNDO DE INVESTIMENTO EM CADEIAS PRODUTIVAS AGROINDUSTRIAIS IMOBILIÁRIO
[4/1340] 49.489.928/0001-70 → A | TANE FZDB FUNDO DE INVESTIMENTO NAS CADEIAS PRODUTIVAS AGROINDUSTRIAIS
[5/1340] 19.419.994/0001-03 → A | 2509 FUNDO DE INVESTIMENTO IMOBILIÁRIO - FII RESPONSABILIDADE LIMITADA
[6/1340] 61.530.579/0001-99 → A | 3M FUNDO DE  INVESTIMENTO IMOBILIÁRIO RESPONSABILIDADE LIMITADA
[7/1340] 20.717.355/0001-03 → A | LA SHOPPING CENTERS II FUNDO DE INVESTIMENTO IMOBILIARIO
[8/1340] 36.885.989/0001-05 → A | ABEL 33 FUNDO DE INVESTIMENTO IMOBILIÁRIO DE RESPONSABILIDADE ILIMITADA
[9/1340] 63.375.304/0001-53 → A | AD SHOPPING FUNDO DE INVESTIMENTO IMOBILIÁRIO RESPONSABILIDADE LIMITADA
[10/1340] 13.873.457/0001-52 → A | FUNDO DE INVESTIMENTO IMOBILIÁRIO RIO BRAVO 

In [13]:
kk['situacao'].value_counts()

situacao
A       943
ERRO    390
C         6
I         1
Name: count, dtype: int64

In [18]:
kk[kk['situacao'] == 'A']

,cnpj,nome,situacao,descricao_status,ticker,total_docs
0,59.680.687/0001-04,TANE FZMG FUNDO DE INVESTIMENTO EM CADEIAS PRO...,A,Ativo com visualização,,41.0
1,43.101.312/0001-02,MEIRINGEN FUNDO DE INVESTIMENTO IMOBILIÁRIO,A,Ativo com visualização,,95.0
2,48.565.175/0001-72,TANE FLEM FUNDO DE INVESTIMENTO EM CADEIAS PRO...,A,Ativo com visualização,FIAGRO FLEM,156.0
3,49.489.928/0001-70,TANE FZDB FUNDO DE INVESTIMENTO NAS CADEIAS PR...,A,Ativo com visualização,FIAGRO FZDB,152.0
4,19.419.994/0001-03,2509 FUNDO DE INVESTIMENTO IMOBILIÁRIO - FII R...,A,Ativo com visualização,,245.0
...,...,...,...,...,...,...
1330,45.188.066/0001-95,ZAGROS MULTIESTRATÉGIA FUNDO DE INVESTIMENTO I...,A,Ativo com visualização,FII ZAGH,188.0
1332,63.879.740/0001-60,ZAMOR FUNDO DE INVESTIMENTO IMOBILIÁRIO RESPON...,A,Ativo com visualização,,10.0
1333,53.193.237/0001-00,ÁRTEMIS FUNDO DE INVESTIMENTO IMOBILIÁRIO RESP...,A,Ativo com visualização,FII ARTE,131.0
1334,53.137.434/0001-02,ÍCONE - FUNDO DE INVESTIMENTO IMOBILIÁRIO RESP...,A,Ativo com visualização,FII ICNE,90.0
